### Phase 5: Evaluation & Vergleich (Der Kern der Thesis)
Beweis des Mehrwerts zur Reduktion des **Sequence of Returns Risk**.
*   **Metriken:**
    *   **Maximum Drawdown (MDD):** Wie tief war der maximale Fall? (Wichtigstes Ziel).
    *   **Sharpe Ratio / Sortino Ratio:** Risiko-adjustierte Rendite.
    *   **Calmar Ratio:** Rendite im Verhältnis zum Drawdown.
    *   **Regime-Stabilität:** Wie oft schaltet das Modell um? (LSTMs neigen zum "Overfitting" und Rauschen).

In [ ]:
# --- Central Config ---
import sys; sys.path.insert(0, "../config")
from config_loader import cfg

In [ ]:
import pandas as pd

# Daten aus dem data-Ordner laden
backtesting_results = pd.read_parquet(cfg.data_path("backtesting_results"))
backtesting_transaction_costs = pd.read_parquet(cfg.data_path("backtesting_costs"))
test_df = pd.read_parquet(cfg.data_path("test_data"))

In [ ]:
import numpy as np

# --- Evaluation aus src/ ---
sys.path.insert(0, "..")
from src.backtest.evaluation import evaluate_strategies

# Daten laden
test_df = pd.read_parquet(cfg.data_path("test_data"))
backtesting_results = pd.read_parquet(cfg.data_path("backtesting_results"))

# --- DYNAMISCHE ZUORDNUNG DER SIGNALE ---
signals_to_count = pd.DataFrame(index=test_df.index)
signal_columns = [c for c in test_df.columns if c.endswith('_Signal')]

for sig_col in signal_columns:
    model_name = sig_col.rsplit('_', 1)[0]
    signals_to_count[model_name] = test_df[sig_col]

# Evaluation ausführen
evaluation_table = evaluate_strategies(backtesting_results, signals_to_count, backtesting_transaction_costs)

# Anzeige und Persistierung
print("\n--- UMFASSENDE EVALUATION (DYNAMISCH) ---")
display(evaluation_table)

evaluation_table.to_markdown(cfg.asset_path("evaluation_table"), index=True)
print(f"\nEvaluationstabelle erfolgreich unter {cfg.asset_path('evaluation_table')} persistiert.")

In [ ]:
# --- 06_MCS: Block-Bootstrap Robustness-Check ---

import pandas as pd
import os
from src.backtest.evaluation import run_monte_carlo_simulation
from src.backtest.sorr import build_sorr_scenarios
from src.backtest.plots import plot_mcs_boxplots, plot_mcs_paths, plot_mcs_quantiles

N_SIMULATIONS = cfg.evaluation.mcs.n_paths
BLOCK_SIZE = cfg.evaluation.mcs.block_length
RANDOM_SEED = cfg.evaluation.mcs.random_seed
SIM_YEARS = cfg.evaluation.mcs.sim_years
TRADING_DAYS = cfg.evaluation.mcs.trading_days_per_year
TOTAL_DAYS = SIM_YEARS * TRADING_DAYS

scenarios = build_sorr_scenarios(cfg.backtesting.sorr.scenarios)
daily_rets = backtesting_results.pct_change().dropna()

all_mc_summaries, mcs_paths_collector = run_monte_carlo_simulation(
    daily_rets=daily_rets,
    test_df=test_df,
    scenarios=scenarios,
    n_simulations=N_SIMULATIONS,
    block_size=BLOCK_SIZE,
    random_seed=RANDOM_SEED,
    sim_years=SIM_YEARS,
    trading_days_per_year=TRADING_DAYS,
)

# Boxplots
plot_mcs_boxplots(
    mcs_paths_collector, daily_rets.columns, scenarios, SIM_YEARS,
    save_path_template=os.path.join(str(cfg._base_dir / "assets"), "mcs_boxplot_{}.png"),
)

# MCS DataFrame für Pfade & Quantile
mcs_results = pd.DataFrame(mcs_paths_collector)

scenarios_list = list(vars(cfg.backtesting.sorr.scenarios).keys())
strategies = backtesting_results.columns

# Pfad-Verläufe
plot_mcs_paths(mcs_results, scenarios_list, strategies, cfg.color_map,
               cfg.asset_path("mcs_paths"))

# Quantile
plot_mcs_quantiles(mcs_results, scenarios_list, strategies, TOTAL_DAYS, cfg.color_map,
                   cfg.asset_path("mcs_quantiles"))

# Summary
if all_mc_summaries:
    mc_multi_summary = pd.DataFrame(all_mc_summaries).set_index(['Szenario', 'Strategie'])
    mc_multi_summary.to_markdown(cfg.asset_path("mcs_summary"))
    display(mc_multi_summary)

print("Alle Monte Carlo Simulationen abgeschlossen.")

from IPython.display import Image, display
for sc_name in scenarios_list:
    boxplot_path = cfg.asset_path(f"mcs_boxplot_{sc_name.lower()}")
    display(Image(filename=boxplot_path))
display(Image(filename=cfg.asset_path("mcs_paths")))
display(Image(filename=cfg.asset_path("mcs_quantiles")))

print(mcs_results)

In [ ]:
output_path = cfg.data_path('mcs_data')

# Speichern als Parquet
mcs_results.to_parquet(output_path)

print(f"MCS-Dataframe erfolgreich unter {output_path} gespeichert.")